[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_4_HigherOrder_Operators.ipynb) [![Launch JupyterLab](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hany73-max/PDE-Rover-Engine/main?urlpath=lab/tree/labs/Lab_4_HigherOrder_Operators.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 🛡️ Lab 4: Higher-Order Homogeneous Linear PDEs & Laplace's Equation
### *Mission Control: Why Laplace's Equation $\nabla^2\phi = 0$ Guarantees Zero Local Minima*
**Course:** Applied Partial Differential Equations & Planetary Robotics
**Lecture Notes Reference:** Chapter 1, Section 1.4 — **Notes pp. 34–57** *(PDF pp. 28–51; Laplace's Equation is Example 5 on Notes p. 39 / PDF p. 33)*

---
### 🗺️ Where You Are in the Rover Navigation Pipeline
```
[Labs 1–3: Slopes, Streamlines, & Safety Buffers] ──► 📍 [Lab 4: Laplace's Equation & Harmonic Potentials] ──► [Lab 5: Reducible Operators & Corridor Decay] ──► [Lab 6: Numerical SOR Engine]
```

---
### 🧭 The Rosetta Stone: Connecting Lecture Notes to the Rover Engine

| 1. Lecture Notes Math (PDF pp. 28–51) | 2. Geometric & Physical Meaning | 3. Real Project Function (`rover_path_finder/`) |
| :--- | :--- | :--- |
| $D_x = \frac{\partial}{\partial x}, \quad D_y = \frac{\partial}{\partial y}$ | Algebraic partial differential operators | Factoring higher-order PDEs like polynomials |
| $(D_x^2 + D_y^2)z = 0$ *(Ex. 5, Notes p. 39 / PDF p. 33)* | **Laplace's Equation:** zero net curvature; behaves like a taut soap film with no interior potholes | Governing equation of our navigation potential in [`solvers.py`](rover_path_finder/solvers.py) |
| $m^2 + 1 = 0 \implies m = \pm i$ | Complex conjugate characteristic roots of Laplace's equation | Proves the solution is a smooth **harmonic function** |
| $\nabla^2\phi \approx \frac{\phi_E + \phi_W + \phi_S + \phi_N - 4\phi_C}{h^2} = 0$ | Every interior point equals the exact average of its 4 neighbors | [`_avg(phi)`](rover_path_finder/solvers.py) in `rover_path_finder/solvers.py` |

---
## 1. Intuition First: Why Do We Need a 2nd-Order PDE (Laplace's Equation)?

In Lab 2 we saw that the rover follows the steepest descent streamline of a potential field $\phi(x, y)$ via [`planners.descend()`](rover_path_finder/planners.py).
**But how do we construct $\phi(x, y)$ so that the rover never gets trapped in an artificial pothole (local minimum) on its way to the goal?**

Imagine stretching an elastic membrane over a frame:
* If we hold the start/walls high ($\phi = 1$) and pin the goal low ($\phi = 0$), the membrane settles into mechanical equilibrium:
  $$\frac{\partial^2 \phi}{\partial x^2} + \frac{\partial^2 \phi}{\partial y^2} = 0 \quad \Longleftrightarrow \quad (D_x^2 + D_y^2)\phi = 0$$
* Geometrically, $\frac{\partial^2\phi}{\partial x^2}$ is the curvature along $x$, and $\frac{\partial^2\phi}{\partial y^2}$ is the curvature along $y$.
* For their sum to be zero ($\nabla^2\phi = 0$), the surface **cannot curve upward in both $x$ and $y$ at the same time**! Since a strict local minimum requires $\phi_{xx} > 0$ and $\phi_{yy} > 0$ simultaneously (whose sum would be strictly positive, never zero), **an interior local minimum is mathematically impossible**!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab4

%matplotlib inline
print("Lab 4 Environment Initialized! Ready for Higher-Order Operators & Laplace Theory.")


---
## 2. Complete Section 1.4 Coverage: Homogeneous Operators & All 6 Particular Integral Cases (PDF pp. 28–51)

A linear PDE with constant coefficients is called **homogeneous of order $n$** when every derivative term has the same total order $n$:
$$(a_0 D_x^n + a_1 D_x^{n-1} D_y + \dots + a_n D_y^n) z = f(x, y) \quad \Longleftrightarrow \quad F(D_x, D_y) z = f(x, y)$$

Its general solution is $z = z_1\text{ (Complementary Function C.F.)} + z_2\text{ (Particular Integral P.I.)}$.

---
### Part A: Complementary Function $z_1$ (Notes pp. 35–39 / PDF pp. 29–33)
Substituting the trial wave $z = \phi(y + m x)$ into $F(D_x, D_y)z = 0$ replaces $D_x \to m$ and $D_y \to 1$, yielding the **Auxiliary Equation**:
$$F(m, 1) = a_0 m^n + a_1 m^{n-1} + \dots + a_n = 0$$

| Roots of Auxiliary Equation $F(m, 1) = 0$ | Complementary Function $z_1$ in Lecture Notes |
| :--- | :--- |
| **Case 1: Distinct roots** $m_1 \neq m_2$ *(Notes p. 36 / PDF p. 30)* | $z_1 = \phi_1(y + m_1 x) + \phi_2(y + m_2 x)$ |
| **Case 2: Repeated roots ($k$ times)** $m_1 = \dots = m_k = m$ *(Notes p. 37 / PDF p. 31)* | $z_1 = \phi_1(y + m x) + x\,\phi_2(y + m x) + \dots + x^{k-1}\phi_k(y + m x)$ |
| **Case 3: Complex roots** $m = \alpha \pm i\beta$ *(Notes p. 39 / PDF p. 33)* | $z_1 = \phi_1(y + (\alpha + i\beta)x) + \phi_2(y + (\alpha - i\beta)x)$ |

---
### 📖 Key Worked Example: Laplace's Equation (Example 5, Notes p. 39 / PDF p. 33)
**Problem:** Solve the 2D Laplace equation:
$$\frac{\partial^2 z}{\partial x^2} + \frac{\partial^2 z}{\partial y^2} = 0 \quad \Longleftrightarrow \quad (D_x^2 + D_y^2)z = 0$$
1. **Auxiliary Equation:** Set $D_x = m$ and $D_y = 1$:
   $$m^2 + 1 = 0 \implies m^2 = -1 \implies m = \pm i$$
2. **General Solution:**
   $$z = \phi_1(y + i x) + \phi_2(y - i x)$$
   Choosing complex conjugate harmonic combinations (such as $\sin(\pi x)\sinh(\pi y)$) produces smooth, real-valued harmonic surfaces!

---
### Part B: Particular Integral $z_2 = \frac{1}{F(D_x, D_y)} f(x, y)$ — The General Method & All 6 Short Cases (PDF pp. 34–51)

When the right-hand side $f(x, y) \neq 0$, your lecture notes teach **1 General Method** plus **6 Short Methods** depending on the shape of $f(x, y)$:

#### 🧭 Quick Decision Table for All 6 Particular Integral Cases (PDF pp. 34–51)

| Method in Notes | Right-Hand Side $f(x, y)$ | Fast Rule for $z_2 = \frac{1}{F(D_x, D_y)}f(x,y)$ | What if the Denominator Vanishes ($F = 0$)? |
| :--- | :--- | :--- | :--- |
| **General Method** *(Notes p. 40 / PDF p. 34)* | Any general function (e.g., $\sec^2(x+y)$) | Factor $F = \prod(D_x - m_i D_y)$ and solve each 1st-order factor via Lagrange $\frac{dx}{1} = \frac{dy}{-m} = \frac{du}{f}$ | Always works without exception! |
| **Case 1** *(Notes p. 44 / PDF p. 38)* | $e^{ax + by}$ | Replace $D_x \to a$ and $D_y \to b$: $z_2 = \frac{1}{F(a,b)}e^{ax+by}$ | If $F = (D_x - \frac{a}{b}D_y)^r G$: $z_2 = \frac{1}{G(a,b)}\frac{x^r}{r!}e^{ax+by}$ |
| **Case 2** *(Notes p. 47 / PDF p. 41)* | $\sin(ax+by)$ or $\cos(ax+by)$ | Replace $D_x^2 \to -a^2,\; D_x D_y \to -ab,\; D_y^2 \to -b^2$ | Convert to exponentials via Euler's formula $\frac{e^{i\theta} \pm e^{-i\theta}}{2}$ and apply Case 1's resonance rule |
| **Case 3** *(Notes p. 51 / PDF p. 45)* | Polynomial $x^a y^b$ | Factor out lowest power of $D_x$ (or $D_y$) and expand $\frac{1}{1-\theta} = 1 + \theta + \theta^2 + \dots$ | $\frac{1}{D_x}$ means integrating w.r.t. $x$ |
| **Case 4** *(Notes p. 53 / PDF p. 47)* | $e^{ax+by} V(x,y)$ | **Exponential Shift:** Pull $e^{ax+by}$ outside and shift $D_x \to D_x+a,\; D_y \to D_y+b$ | Apply Case 1, 2, or 3 to $V(x,y)$ after shifting |
| **Cases 5 & 6** *(Notes p. 56 / PDF p. 50)* | Any function $g(ax+by)$ | If $F(a,b)\neq 0$: integrate $g(u)$ $n$ times: $\frac{1}{F(a,b)}\int\dots\int g(u)du^n$ | **Case 6:** For $(bD_x - aD_y)^n z = g(ax+by)$: $z_2 = \frac{x^n}{n!\,b^n}g(ax+by)$ |

---
#### 🔹 The General Method (Factoring + Lagrange — Notes pp. 40–44 / PDF pp. 34–38)
* **How it works:** Factor $F(D_x, D_y) = (D_x - m_1 D_y)(D_x - m_2 D_y)$ and let $u_1 = (D_x - m_1 D_y)z$. Then $(D_x - m_2 D_y)u_1 = f(x, y)$ is a 1st-order PDE solved by Lagrange's auxiliary equations $\frac{dx}{1} = \frac{dy}{-m_2} = \frac{du_1}{f(x,y)}$ (from Lab 2!), and we repeat for each factor.
* **📖 Worked Example 1 (Notes pp. 41–43 / PDF pp. 35–37):** For $(D_x^2 - D_y^2)z = \sec^2(x + y)$, factoring $(D_x + D_y)(D_x - D_y)z = \sec^2(x + y)$ and integrating via Lagrange yields $\boxed{z_2 = \frac{x}{2}\tan(x + y)}$.

---
#### 🔹 Case 1: Exponential $f(x, y) = e^{ax + by}$ & The Resonance Rule $F(a, b) = 0$ (Notes pp. 44–47 / PDF pp. 38–41)
* **a) Non-resonant case ($F(a, b) \neq 0$):** Substitute $D_x \to a,\; D_y \to b$:
  $$\boxed{z_2 = \frac{1}{F(a, b)}e^{ax + by}}$$
  * **Example 2 (Notes p. 45 / PDF p. 39):** For $(D_x^2 - D_x D_y - 6D_y^2)z = e^{2x - 3y}$, $a = 2, b = -3 \implies F(2, -3) = 4 - (-6) - 6(9) = -44 \neq 0 \implies \boxed{z_2 = -\frac{1}{44}e^{2x - 3y}}$.
* **b) Resonant case ($F(a, b) = 0$ — Notes pp. 45–47 / PDF pp. 39–41):**
  When $F(a, b) = 0$, $\left(D_x - \frac{a}{b}D_y\right)$ is a factor of $F(D_x, D_y)$ of multiplicity $r$:
  $$F(D_x, D_y) = \left(D_x - \frac{a}{b}D_y\right)^r G(D_x, D_y), \qquad G(a, b) \neq 0$$
  Evaluate $(a, b)$ in the non-zero factor $G(a, b)$, and replace the resonant factor $\frac{1}{(D_x - \frac{a}{b}D_y)^r}$ by **multiplying by $\frac{x^r}{r!}$**:
  $$\boxed{z_2 = \frac{1}{G(a, b)} \frac{x^r}{r!} e^{ax + by}}$$
  * **📖 Worked Example 3 (Notes p. 46 / PDF p. 40):** Solve $(D_x^2 - D_x D_y - 6D_y^2)z = e^{3x + y}$.
    1. Here $a = 3, b = 1 \implies F(3, 1) = 9 - 3 - 6 = 0$!
    2. Factor the operator: $D_x^2 - D_x D_y - 6D_y^2 = (D_x - 3D_y)^1 (D_x + 2D_y)$.
    3. The resonant factor has multiplicity $r = 1$, and $G(D_x, D_y) = D_x + 2D_y \implies G(3, 1) = 3 + 2(1) = 5$.
    4. Apply the resonance formula: $\boxed{z_2 = \frac{1}{5}\frac{x^1}{1!}e^{3x + y} = \frac{x}{5}e^{3x + y}}$.

> 📌 **PDF Page 43 Typo Alert (Notes p. 49, Example 5):**
> At the bottom of Notes page 49 (PDF page 43), when evaluating $F = D_x^2 - 3D_x D_y + 2D_y^2$ on $e^{2x+3y}$ ($a=2, b=3$), the notes write $F(1,1) = 4 - 18 + 18 = 4$. The arithmetic $2^2 - 3(2)(3) + 2(3)^2 = 4 - 18 + 18 = 4$ is completely right for $(2, 3)$, so $F(1,1)$ is just a typo for $\boxed{F(2, 3) = 4}$!

---
#### 🔹 Case 2: Sine & Cosine $f(x, y) = \sin(ax+by)$ or $\cos(ax+by)$ (Notes pp. 47–51 / PDF pp. 41–45)
* **a) When the 2nd-order denominator is non-zero:** Substitute:
  $$\boxed{D_x^2 \to -a^2, \qquad D_x D_y \to -ab, \qquad D_y^2 \to -b^2}$$
  *(Watch the minus signs! For $\sin(2x - 3y)$, $a = 2, b = -3$, so $D_x^2 \to -(2^2) = -4$, $D_y^2 \to -((-3)^2) = -9$, and $D_x D_y \to -(2)(-3) = +6$).*
* **b) When the denominator vanishes (Resonance — Example 6, Notes pp. 50–51 / PDF pp. 44–45):**
  Rewrite $\cos\theta = \frac{e^{i\theta} + e^{-i\theta}}{2}$ (or $\sin\theta = \frac{e^{i\theta} - e^{-i\theta}}{2i}$) and apply Case 1's resonance rule!

---
#### 🔹 Case 3: Polynomials $f(x, y) = x^a y^b$ via Binomial Series Expansion (Notes pp. 51–53 / PDF pp. 45–47)
* **How it works:** Factor out the lowest power of $D_x$ (or $D_y$) from the denominator so it takes the form $D_x^k [1 - \theta(D_x, D_y)]$, then expand $(1 - \theta)^{-1}$ as a geometric series:
  $$\boxed{\frac{1}{1 - \theta} = 1 + \theta + \theta^2 + \theta^3 + \dots, \qquad \frac{1}{1 + \theta} = 1 - \theta + \theta^2 - \theta^3 + \dots}$$
  Because $x^a y^b$ is a polynomial, its higher derivatives vanish after finitely many terms! Remember that $\frac{1}{D_x}$ means **integrating w.r.t. $x$**.
* **📖 Worked Example 7 (Notes p. 52 / PDF p. 46):** Find $z_2$ for $(D_x^2 - 2D_x D_y)z = x^3 y$.
  1. Factor out $D_x^2$:
     $$z_2 = \frac{1}{D_x^2\left(1 - \frac{2D_y}{D_x}\right)}(x^3 y) = \frac{1}{D_x^2}\left(1 + \frac{2D_y}{D_x} + \dots\right)(x^3 y)$$
  2. Since $D_y(x^3 y) = x^3$ and $D_y^2(x^3 y) = 0$, all higher terms drop out:
     $$z_2 = \frac{1}{D_x^2}\left[x^3 y + \frac{2}{D_x}(x^3)\right] = \frac{1}{D_x^2}\left(x^3 y + \frac{x^4}{2}\right) = \frac{1}{D_x}\left(\frac{x^4 y}{4} + \frac{x^5}{10}\right) = \boxed{\frac{x^5 y}{20} + \frac{x^6}{60}}$$

---
#### 🔹 Case 4: Exponential Shift $f(x, y) = e^{ax + by} V(x, y)$ (Notes pp. 53–55 / PDF pp. 47–50)
* **Exponential Shift Theorem:** Pull $e^{ax + by}$ to the front of the operator by shifting every $D_x \to D_x + a$ and $D_y \to D_y + b$:
  $$\boxed{\frac{1}{F(D_x, D_y)}\left[e^{ax + by} V(x, y)\right] = e^{ax + by} \frac{1}{F(D_x + a, D_y + b)} V(x, y)}$$
* **📖 Worked Example 10 (Notes p. 54 / PDF p. 48):** Solve $(D_x - D_y)^2 z = x e^{x + y}$.
  1. Pull $e^{x + y}$ ($a = 1, b = 1$) out front and shift $D_x \to D_x + 1,\; D_y \to D_y + 1$:
     $$(D_x + 1) - (D_y + 1) = D_x - D_y \implies z_2 = e^{x+y}\frac{1}{(D_x - D_y)^2}(x)$$
  2. Since $D_y(x) = 0$, this reduces to integrating $x$ twice w.r.t. $x$:
     $$\boxed{z_2 = e^{x + y}\frac{1}{D_x^2}(x) = \frac{x^3}{6}e^{x + y}}$$

---
#### 🔹 Cases 5 & 6: Arbitrary Function of a Linear Combination $f(x, y) = g(ax + by)$ (Notes pp. 56–57 / PDF pp. 50–51)
* **Case 5 ($F(a, b) \neq 0$ — Notes p. 56 / PDF p. 50):** Set $u = ax + by$ and integrate $g(u)$ $n$ times (where $n$ is the order of $F$):
  $$\boxed{z_2 = \frac{1}{F(a, b)} \underbrace{\int \dots \int}_{n\text{ times}} g(u)\,du^n}$$
* **Case 6 ($F(a, b) = 0$ where $F(D_x, D_y) = (b D_x - a D_y)^n$ — Notes pp. 56–57 / PDF pp. 50–51):**
  $$\boxed{z_2 = \frac{x^n}{n!\,b^n}\,g(ax + by)}$$
  * **📖 Worked Examples 14 & 15 (Notes p. 57 / PDF p. 51):**
    * **Ex. 14:** $(2D_x - 3D_y)^2 z = \tan(3x + 2y)$ ($a = 3, b = 2, n = 2$) $\implies \boxed{z_2 = \frac{x^2}{2!\cdot 2^2}\tan(3x + 2y) = \frac{x^2}{8}\tan(3x + 2y)}$.
    * **Ex. 15:** $(D_x - 2D_y)^3 z = (2x + y)^2$ ($a = 2, b = 1, n = 3$) $\implies \boxed{z_2 = \frac{x^3}{3!\cdot 1^3}(2x + y)^2 = \frac{x^3}{6}(2x + y)^2}$.

> 📌 **PDF Page 51 Typo Alert (Notes p. 57, Example 14):**
> In the solution of Example 14, the notes write $F(D_x, D_x) = (2D_x - 3D_y)^2$ with $D_x$ repeated twice in the argument list; that is a printing typo for $\boxed{F(D_x, D_y)}$.


---
## 3. 📝 Paper & Pen Challenge 1 (Laplace's Equation — Ex. 5, Notes p. 39 / PDF p. 33)

**Problem:** Find the characteristic roots $m_1, m_2$ of Laplace's equation:
$$(D_x^2 + D_y^2)z = 0$$

In Python, the imaginary unit $i = \sqrt{-1}$ is written `1j`, and $-i$ is written `-1j`.

<details>
<summary>💡 <b>Hint (Solving the Auxiliary Equation)</b></summary>

Setting $D_x = m$ and $D_y = 1$ gives $m^2 + 1 = 0 \implies m = \pm i$, which in Python are `1j` and `-1j`.
</details>


In [ ]:
# GRADED CELL: Paper Challenge 1
### START CODE HERE ###
m1 = 1j
m2 = -1j
### END CODE HERE ###

# Run numerical verification test:
test_lab4.check_paper_challenge_1(m1, m2)


---
## 4. 📝 Paper & Pen Challenge 2 (Particular Integral — Ex. 4, Notes p. 48 / PDF p. 42)

**Problem (Notes p. 48, Example 4 / PDF p. 42):** Find the Particular Integral $z_2$ of:
$$(D_x^2 - D_x D_y - 6 D_y^2)z = \sin(2x - 3y)$$

**Work through these steps on paper:**
1. Identify $a = 2$ and $b = -3$ from $\sin(2x - 3y)$.
2. Substitute $D_x^2 \to -a^2$, $D_x D_y \to -ab$, and $D_y^2 \to -b^2$ into the operator $(D_x^2 - D_x D_y - 6 D_y^2)$.
3. Evaluate the numerical denominator and write the resulting formula for $z_2$.

<details>
<summary>💡 <b>Hint 1 (Evaluating the Substitutions)</b></summary>

With $a = 2$ and $b = -3$: $D_x^2 \to -4$, $D_x D_y \to +6$, and $D_y^2 \to -9$.
</details>

<details>
<summary>💡 <b>Hint 2 (Evaluating the Operator Denominator)</b></summary>

Denominator $= (-4) - (6) - 6(-9) = -10 + 54 = 44$, so $z_2 = \frac{1}{44}\sin(2x - 3y)$.
</details>


In [ ]:
# GRADED CELL: Particular Integral Challenge
### START CODE HERE ###
student_pi = "(1/44)*sin(2x - 3y)"
### END CODE HERE ###

# Run numerical verification test:
test_lab4.check_paper_challenge_2(student_pi)


---
## 5. 💻 Python Implementation: Characteristic Operator Root Solver

To solve any homogeneous operator $(a_0 D_x^n + a_1 D_x^{n-1}D_y + \dots + a_n D_y^n)z = 0$ in Python, we pass the polynomial coefficient list `[a_0, a_1, ..., a_n]` into `np.roots(coeffs)`. Complete `solve_operator_roots` below.


In [ ]:
# GRADED FUNCTION: solve_operator_roots
def solve_operator_roots(coeffs: list[float]) -> list[complex]:
    # Finds the characteristic roots of the auxiliary equation F(m, 1) = 0
    ### START CODE HERE ###
    roots = np.roots(coeffs)
    ### END CODE HERE ###
    return list(roots)

# Run automated unit test:
test_lab4.test_characteristic_roots_solver(solve_operator_roots)


---
## 6. 🎨 Bridge to Lab 6: How Analytical Laplace Solutions Connect to `solvers._avg(phi)`

You might wonder: **If we know the analytical solution $z = \phi_1(y + ix) + \phi_2(y - ix)$, why do we need a numerical solver in [`solvers.py`](rover_path_finder/solvers.py)?**
* On a simple empty square, we can write a closed-form harmonic function like $\phi(x, y) = \frac{\sin(\pi x)\sinh(\pi y)}{\sinh(\pi)}$.
* On Mars, however, the map contains **thousands of irregular rock/crater pixels**—no single formula can match 10,000 arbitrary obstacle boundaries!
* When we approximate second derivatives on a uniform pixel grid ($\Delta x = \Delta y = h$):
  $$\frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} \approx \frac{\phi_{i,j+1} + \phi_{i,j-1} + \phi_{i+1,j} + \phi_{i-1,j} - 4\phi_{i,j}}{h^2} = 0$$
  Laplace's equation $(D_x^2 + D_y^2)\phi = 0$ states that **every interior point equals the average of its 4 neighbors** (`_avg(phi)` in [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py))!

Run the cell below to see the analytical harmonic surface alongside the 4-neighbor average check using `solvers._avg`:


In [ ]:
from rover_path_finder.solvers import _avg

x = np.linspace(0, 1, 50)
y = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.pi * X) * np.sinh(np.pi * Y) / np.sinh(np.pi)

neighbor_diff = np.abs(_avg(Z)[1:-1, 1:-1] - Z[1:-1, 1:-1])

fig = plt.figure(figsize=(13, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.9)
ax1.set_title(r"1. Harmonic Surface $\nabla^2\phi = 0$ (Zero Local Minima)", fontsize=11)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel(r"$\phi(x, y)$")
plt.colorbar(surf, ax=ax1, shrink=0.55)

ax2 = fig.add_subplot(1, 2, 2)
im = ax2.imshow(neighbor_diff, origin='lower', extent=[x[1], x[-2], y[1], y[-2]], cmap='magma')
plt.colorbar(im, ax=ax2, label=r"$|\mathrm{\_avg}(\phi) - \phi|$")
ax2.set_title("2. Bridge to solvers.py: 4-Neighbor Average Residual", fontsize=11)
ax2.set_xlabel("x")
ax2.set_ylabel("y")

plt.tight_layout()
plt.show()

print(f"Max difference between analytical harmonic surface and _avg(Z): {neighbor_diff.max():.6f}")
print("Lab 4 Takeaways:")
print("  1) Laplace's equation (D_x^2 + D_y^2)z = 0 has complex roots m = ±i and smooth harmonic solutions.")
print("  2) Harmonic functions obey the 4-neighbor Mean Value Property (_avg in solvers.py), preventing local traps.")
